In [63]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [64]:
DATA_DIR = Path("processed_data")
DATA_DIR.mkdir(exist_ok=True)

def save_dataset(df, name, save_csv=True):
    parquet_path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(parquet_path, index=False)

    if save_csv:
        csv_path = DATA_DIR / f"{name}.csv"
        df.to_csv(csv_path, index=False)

    print(f"Saved dataset: {name}")

def load_dataset(name):
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)

In [65]:
bond_df = load_dataset("bond_panel")

In [66]:
SHOCKS_DIR = Path("Bond_Research") / "jkshocks_update_ecb-main"

shock_df = pd.read_csv(
    SHOCKS_DIR / "shocks_ecb_mpd_me_m.csv"
)

In [67]:
shock_df.head()
shock_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   year        322 non-null    int64  
 1   month       322 non-null    int64  
 2   pc1_hf      322 non-null    float64
 3   STOXX50_hf  322 non-null    float64
 4   MP_pm       322 non-null    float64
 5   CBI_pm      322 non-null    float64
 6   MP_median   322 non-null    float64
 7   CBI_median  322 non-null    float64
dtypes: float64(6), int64(2)
memory usage: 20.3 KB


Date adjustments

In [68]:
shock_df["Date"] = pd.to_datetime(
    dict(
        year=shock_df["year"],
        month=shock_df["month"],
        day=1
    )
)

In [69]:
shock_df = shock_df[
    [
        "Date",
        "MP_median",
        "CBI_median"
    ]
].copy()

In [70]:
shock_df = shock_df.rename(
    columns={
        "MP_median": "MP_Shock",
        "CBI_median": "CBI_Shock"
    }
)

In [71]:
shock_df.head()

,Date,MP_Shock,CBI_Shock
0,1999-01-01,0.029146,-0.063111
1,1999-02-01,-0.005565,0.005565
2,1999-03-01,-0.005922,0.003238
3,1999-04-01,-0.006799,-0.003306
4,1999-05-01,0.015431,-0.009366


In [72]:
save_dataset(
    shock_df,
    "ecb_shocks_monthly"
)

Saved dataset: ecb_shocks_monthly


In [73]:
monthly_bond_df = bond_df.copy()

monthly_bond_df["Month"] = (
    monthly_bond_df["Date"]
    .dt.to_period("M")
)

We take the last avb daily observation. 

In [74]:
monthly_bond_df = (
    monthly_bond_df
    .sort_values("Date")
    .groupby(
        ["Country", "Maturity", "Month"],
        as_index=False
    )
    .last()
)

In [75]:
monthly_bond_df["Date"] = (
    monthly_bond_df["Month"]
    .dt.to_timestamp()
)

In [76]:
monthly_bond_df = monthly_bond_df.drop(
    columns="Month"
)

In [77]:
monthly_bond_df = monthly_bond_df[
    [
        "Date",
        "Country",
        "Maturity",
        "Yield",
        "BundYield",
        "Spread"
    ]
]
monthly_bond_df.head()

,Date,Country,Maturity,Yield,BundYield,Spread
0,1999-01-01,Austria,10Y,NaN,3.632,NaN
1,1999-02-01,Austria,10Y,NaN,4.011,NaN
2,1999-03-01,Austria,10Y,NaN,4.001,NaN
3,1999-04-01,Austria,10Y,NaN,3.868,NaN
4,1999-05-01,Austria,10Y,4.314,4.116,0.198


In [78]:
monthly_bond_df.groupby(
    ["Country", "Maturity"]
).size()

Country      Maturity
Austria      10Y         333
             2Y          333
             5Y          333
Belgium      10Y         333
             2Y          333
             5Y          333
France       10Y         333
             2Y          333
             5Y          333
Germany      10Y         333
             2Y          333
             5Y          333
Greece       10Y         333
             2Y          333
             5Y          333
Ireland      10Y         333
             2Y          333
             5Y          333
Italy        10Y         333
             2Y          333
             5Y          333
Netherlands  10Y         333
             2Y          333
             5Y          333
Portugal     10Y         333
             2Y          333
             5Y          333
Spain        10Y         333
             2Y          333
             5Y          333
dtype: int64

In [79]:
monthly_bond_df["Date"].min()


Timestamp('1999-01-01 00:00:00')

In [80]:
monthly_bond_df["Date"].max()

Timestamp('2026-09-01 00:00:00')

In [81]:
lp_base_df = monthly_bond_df.merge(
    shock_df,
    on="Date",
    how="left"
)

In [82]:
lp_base_df.head()

,Date,Country,Maturity,Yield,BundYield,Spread,MP_Shock,CBI_Shock
0,1999-01-01,Austria,10Y,NaN,3.632,NaN,0.029146,-0.063111
1,1999-02-01,Austria,10Y,NaN,4.011,NaN,-0.005565,0.005565
2,1999-03-01,Austria,10Y,NaN,4.001,NaN,-0.005922,0.003238
3,1999-04-01,Austria,10Y,NaN,3.868,NaN,-0.006799,-0.003306
4,1999-05-01,Austria,10Y,4.314,4.116,0.198,0.015431,-0.009366


In [83]:
lp_base_df[
    ["MP_Shock", "CBI_Shock"]
].isna().sum()

MP_Shock     330
CBI_Shock    330
dtype: int64

Date check

In [84]:
shock_df["Date"].min(), shock_df["Date"].max()

(Timestamp('1999-01-01 00:00:00'), Timestamp('2025-10-01 00:00:00'))

In [85]:
monthly_bond_df["Date"].min(), monthly_bond_df["Date"].max()

(Timestamp('1999-01-01 00:00:00'), Timestamp('2026-09-01 00:00:00'))

In [86]:
save_dataset(
    lp_base_df,
    "lp_base_monthly"
)

Saved dataset: lp_base_monthly


In [87]:
missing_dates = (
    lp_base_df.loc[
        lp_base_df["MP_Shock"].isna(),
        "Date"
    ]
    .drop_duplicates()
    .sort_values()
)

print(missing_dates.to_string(index=False))

2025-11-01
2025-12-01
2026-01-01
2026-02-01
2026-03-01
2026-04-01
2026-05-01
2026-06-01
2026-07-01
2026-08-01
2026-09-01


In [88]:
missing_check = pd.DataFrame({
    "Date": missing_dates
})

missing_check = missing_check.merge(
    shock_df,
    on="Date",
    how="left"
)

print(missing_check)

         Date  MP_Shock  CBI_Shock
0  2025-11-01       NaN        NaN
1  2025-12-01       NaN        NaN
2  2026-01-01       NaN        NaN
3  2026-02-01       NaN        NaN
4  2026-03-01       NaN        NaN
5  2026-04-01       NaN        NaN
6  2026-05-01       NaN        NaN
7  2026-06-01       NaN        NaN
8  2026-07-01       NaN        NaN
9  2026-08-01       NaN        NaN
10 2026-09-01       NaN        NaN


In [89]:
print(monthly_bond_df["Date"].head())
print(shock_df["Date"].head())

print(monthly_bond_df["Date"].dtype)
print(shock_df["Date"].dtype)

0   1999-01-01
1   1999-02-01
2   1999-03-01
3   1999-04-01
4   1999-05-01
Name: Date, dtype: datetime64[us]
0   1999-01-01
1   1999-02-01
2   1999-03-01
3   1999-04-01
4   1999-05-01
Name: Date, dtype: datetime64[us]
datetime64[us]
datetime64[us]


In [90]:
lp_df = lp_base_df[
    lp_base_df["Date"] <= shock_df["Date"].max()
].copy()

In [91]:
lp_df[
    ["MP_Shock", "CBI_Shock"]
].isna().sum()

MP_Shock     0
CBI_Shock    0
dtype: int64

In [92]:
save_dataset(
    lp_df,
    "lp_estimation_sample"
)

Saved dataset: lp_estimation_sample


In [93]:
save_dataset(
    monthly_bond_df,
    "bond_panel_monthly"
)

Saved dataset: bond_panel_monthly
